In [ ]:
from utils import *

In [ ]:
root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

file_paths = traverse_directory(root_directory)
#main_files = [path for path in file_paths if ("subgraph" not in path and "test" not in path and "all_country_capital" in path)]

root_directory = "/nfs/datz/olmo_models/new_processed_outputs/"

all_files = [file for file in file_paths if "test" not in file]

models_data = read_all_files(all_files)

for model_name, relations in models_data.items():
    print(f"Model: {model_name}")
    for relation_name, data_entries in relations.items():
        print(f"  Relation: {relation_name}, Number of Entries: {len(data_entries)}")
        
sorted_models = sorted(
        models_data.keys(), 
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )

In [ ]:
model_name = "allenai/OLMo-7B-0424-hf"

model = load_models(model_name, device='cuda:6', checkpoint="main")

In [ ]:
heatmaps_dict = {}

theta = 0.1

for model_name in sorted_models:
    relations_data = models_data[model_name]

    print(model_name)
    g_total = 0
    general_heatmap = np.zeros((32, 32), dtype=float)
    e_total = 0
    entity_heatmap = np.zeros((32, 32), dtype=float)

    relation_answer_heatmaps = {}
    relation_answer_with_specific = {}

    for relation_name, entries in relations_data.items():

        relation_answer_heatmap = np.zeros((32, 32), dtype=float)
        relation_answer_total = 0
        answer_specific_heatmaps = {}

        for idx, entry in enumerate(entries):

            # Ensure subj_token_span is not None
            if entry["subj_token_span"] is None:
                entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

            # Update general heatmap
            general_heatmap += np.mean(entry['attnheads_contribution_maps'], axis=0) #added [:entry['answer_token_span'][0]] because I dont need the contributions from the answer token itself
            g_total += 1

            # Update entity heatmap for subject tokens
            one_data_map = np.zeros((32, 32), dtype=float)
            for t in entry["subj_token_span"]:
                # if t == 0:
                #     e_total -= 1
                #     continue
                one_data_map += entry['attnheads_contribution_maps'][t]#[t - 1]
            entity_heatmap += one_data_map
            e_total += len(entry["subj_token_span"])

            # Update entity heatmap and relation answer heatmap for answer tokens
            one_data_map = np.zeros((32, 32), dtype=float)
            for t in entry["answer_token_span"]:
                one_data_map += entry['attnheads_contribution_maps'][t - 1]
            entity_heatmap += one_data_map
            relation_answer_heatmap += one_data_map

            e_total += len(entry["answer_token_span"])
            relation_answer_total += len(entry["answer_token_span"])

            # Store answer-specific heatmap
            answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
            answer_specific_heatmaps[idx] = answer_specific_heatmap  > theta

        relation_answer_heatmap /= relation_answer_total
        relation_answer_heatmaps[f"{relation_name}"] = relation_answer_heatmap > theta
        relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

    # Normalize heatmaps
    general_heatmap /= g_total
    entity_heatmap /= e_total

    # Store heatmaps in the dictionary
    heatmaps_dict[model_name] = {
        'general_heatmap': general_heatmap > theta,
        'entity_heatmap': entity_heatmap > theta,
        'relation_answer_heatmaps': relation_answer_heatmaps,
        'relation_answer_with_specific': relation_answer_with_specific
    }

In [ ]:
def calculate_consistency(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        if model_name == 'main':
            continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "relation_answer_heatmaps":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "relation_answer_with_specific":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "entity_heatmap":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "general_heatmap":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results


In [ ]:
def plot_iou_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_ious.append(iou_results[model_name]['general_heatmap']['iou'])
        entity_ious.append(iou_results[model_name]['entity_heatmap']['iou'])
        answer_all_ious.append(iou_results[model_name]['relation_answer_heatmaps']['iou'])
        answer_ious.append(iou_results[model_name]['relation_answer_with_specific']['iou'])


    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_ious, label="General Heatmap IoU", marker='o', linestyle='-')
    plt.plot(model_names, entity_ious, label="Entity Heatmap IoU", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_ious, label="Relation Answer Heatmap IoU", marker='x', linestyle='-.')
    plt.plot(model_names, answer_ious, label="Answer Specific Heatmap IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_intersection_union_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )
    general_intersections = []
    general_unions = []
    entity_intersections = []
    entity_unions = []
    answer_all_intersections = []
    answer_all_unions = []
    answer_intersections = []
    answer_unions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_intersections.append(iou_results[model_name]['general_heatmap']['intersection'])
        general_unions.append(iou_results[model_name]['general_heatmap']['union'])
        entity_intersections.append(iou_results[model_name]['entity_heatmap']['intersection'])
        entity_unions.append(iou_results[model_name]['entity_heatmap']['union'])
        answer_all_intersections.append(iou_results[model_name]['relation_answer_heatmaps']['intersection'])
        answer_all_unions.append(iou_results[model_name]['relation_answer_heatmaps']['union'])
        answer_intersections.append(iou_results[model_name]['relation_answer_with_specific']['intersection'])
        answer_unions.append(iou_results[model_name]['relation_answer_with_specific']['union'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_intersections, label="General Heatmap Intersection", marker='o', linestyle='-')
    plt.plot(model_names, entity_intersections, label="Entity Heatmap Intersection", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_intersections, label="Relation Answer Heatmap Intersection", marker='x', linestyle='-.')
    plt.plot(model_names, answer_intersections, label="Answer Specific Heatmap Intersection", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Intersection Area")
    plt.title(f"Intersection Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Plotting Unions
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_unions, label="General Heatmap Union", marker='o', linestyle='-')
    plt.plot(model_names, entity_unions, label="Entity Heatmap Union", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_unions, label="Relation Answer Heatmap Union", marker='x', linestyle='-.')
    plt.plot(model_names, answer_unions, label="Answer Specific Heatmap Union", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Union Area")
    plt.title(f"Union Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
overlap = calculate_consistency(heatmaps_dict, relation_name, fact_idx)
plot_iou_multiple(overlap, relation_name, sent)
plot_intersection_union_multiple(overlap, relation_name, sent)
#plot_all_count_proportion_overlap_multiple(overlap, relation_name, sent)

In [ ]:
relation_name = "books_written"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
overlap = calculate_consistency(heatmaps_dict, relation_name, fact_idx)
plot_iou_multiple(overlap, relation_name, sent)
plot_intersection_union_multiple(overlap, relation_name, sent)
#plot_all_count_proportion_overlap_multiple(overlap, relation_name, sent)

### Proper Heads

In [ ]:
def calculate_proper_heads(heatmaps_dict):
    """
    Calculates proper entity heads, proper relation answer heads, 
    and proper answer specific heads for each model in heatmaps_dict.

    Args:
        heatmaps_dict (dict): A dictionary where each key is a model name and 
                              the value is a dictionary with heatmaps.

    Returns:
        dict: A dictionary containing proper heads for each model.
    """
    proper_heads_dict = {}

    for model_name, heatmaps in heatmaps_dict.items():
        # Extract heatmaps
        general_heads = heatmaps['general_heatmap']
        entity_heads = heatmaps['entity_heatmap']
        relation_answer_heads = heatmaps['relation_answer_heatmaps']
        answer_specific_heads = heatmaps['relation_answer_with_specific']

        # Calculate proper heads
        proper_entity_heads = np.logical_and(entity_heads, np.logical_not(general_heads))
        
        # Initialize dictionaries to store results for relations and specific answers
        proper_relation_answer_heads = {}
        proper_answer_specific_heads = {}

        # Calculate proper relation answer heads per relation
        for relation, relation_heads in relation_answer_heads.items():
            proper_relation_answer_heads[relation] = np.logical_and(
                relation_heads,
                np.logical_not(np.logical_or(proper_entity_heads, general_heads))
            )

        # Calculate proper answer specific heads per relation and specific answer
        for relation, specific_answers in answer_specific_heads.items():
            proper_answer_specific_heads[relation] = {}
            for answer_id, specific_heads in specific_answers.items():
                proper_answer_specific_heads[relation][answer_id] = np.logical_and(
                    specific_heads,
                    np.logical_not(
                        np.logical_or(
                            proper_relation_answer_heads[relation],
                            np.logical_or(proper_entity_heads, general_heads)
                        )
                    )
                )

        # Store results in the output dictionary
        proper_heads_dict[model_name] = {
            'proper_general_heads': general_heads,
            'proper_entity_heads': proper_entity_heads,
            'proper_relation_answer_heads': proper_relation_answer_heads,
            'proper_answer_specific_heads': proper_answer_specific_heads
        }

    return proper_heads_dict

In [ ]:
prop = calculate_proper_heads(heatmaps_dict)

In [ ]:
prop['step5000-tokens20B']['proper_relation_answer_heads']['country_capital_city'].sum()

In [ ]:
def calculate_consistency_proper_heads(heatmaps_dict, relation_name, fact_idx):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    # Iterate through each model (excluding 'main') and each heatmap type
    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU results for this model
        model_ious = {}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Index by relation_name for relation_answer_heatmaps
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]
            elif heatmap_type == "proper_answer_specific_heads":
                # Index by relation_name and fact_idx for relation_answer_with_specific
                main_heatmap = main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_heatmap = model_data[heatmap_type][relation_name][fact_idx]
            elif heatmap_type == "proper_entity_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            elif heatmap_type == "proper_general_heads":
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]
            else:
                continue

            # Calculate IoU
            intersection = np.logical_and(main_heatmap, model_heatmap)
            union = np.logical_or(main_heatmap, model_heatmap)

            area_of_intersection = np.sum(intersection)
            area_of_union = np.sum(union)

            # Avoid division by zero
            iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

            # Store IoU result and statistics
            model_ious[heatmap_type] = {
                'iou': iou,
                'head_count': model_heatmap.sum(),
                'intersection': area_of_intersection,
                'union': area_of_union
            }

        # Add this model's IoU results to the main dictionary
        iou_results[model_name] = model_ious

    return iou_results

In [ ]:
def plot_proper_heads_iou_multiple(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []
    
    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])
        
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Plotting
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_ious, label="Proper General Heads IoU", marker='o', linestyle='-')
    plt.plot(model_names, entity_ious, label="Proper Entity Heads IoU", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_ious, label="Proper Relation Answer Heads IoU", marker='x', linestyle='-.')
    plt.plot(model_names, answer_ious, label="Proper Answer Specific Heads IoU", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper Heads IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_head_count, label="Proper General Heads Count", marker='o', linestyle='-')
    plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count", marker='x', linestyle='-.')
    plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("IoU")
    plt.title(f"Proper Heads IoU Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    
def plot_proper_heads_counts_intersection(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_intersections = []
    entity_intersections = []
    answer_all_intersections = []
    answer_intersections = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
        entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
        answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
        answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_intersections, label="Proper General Heads Intersections", marker='o', linestyle='-')
    plt.plot(model_names, entity_intersections, label="Proper Entity Heads Intersections", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Heads Intersections", marker='x', linestyle='-.')
    plt.plot(model_names, answer_intersections, label="Proper Answer Specific Heads Intersections", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Intersection Counts")
    plt.title(f"Proper Heads Intersections Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

def plot_proper_heads_counts_union(iou_results, relation_name, fact_idx):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda x: int(x.split('step')[1].split('-')[0])
    )

    general_unions = []
    entity_unions = []
    answer_all_unions = []
    answer_unions = []

    # Populate lists for each heatmap type
    for model_name in model_names:
        general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
        entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
        answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
        answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

    # Plotting Intersections
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_unions, label="Proper General Heads Unions", marker='o', linestyle='-')
    plt.plot(model_names, entity_unions, label="Proper Entity Heads Unions", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Heads Unions", marker='x', linestyle='-.')
    plt.plot(model_names, answer_unions, label="Proper Answer Specific Heads Unions", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Union Counts")
    plt.title(f"Proper Heads Unions Across Models for Relation: {relation_name}, Fact: {fact_idx}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
relation_name = "country_capital_city"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(heatmaps_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
relation_name = "books_written"
fact_idx = 1
sent = models_data['main'][relation_name][fact_idx]['sentence']
proper_heads = calculate_proper_heads(heatmaps_dict)
overlap = calculate_consistency_proper_heads(proper_heads, relation_name, fact_idx)
plot_proper_heads_iou_multiple(overlap, relation_name, sent)
plot_proper_heads_counts_intersection(overlap, relation_name, sent)
plot_proper_heads_counts_union(overlap, relation_name, sent)

In [ ]:
def calculate_consistency_proper_heads_all_facts(heatmaps_dict, relation_name):
    iou_results = {}

    # Retrieve the 'main' model's heatmaps
    main_heatmaps = heatmaps_dict['main']

    for model_name, model_data in heatmaps_dict.items():
        #if model_name == 'main':
        #    continue

        # Store IoU, intersection, and union results for this model
        model_metrics = {heatmap_type: {'iou': [], 'head_count': [], 'intersection': [], 'union': []}
                         for heatmap_type in main_heatmaps.keys()}

        for heatmap_type in main_heatmaps.keys():
            if heatmap_type == "proper_relation_answer_heads":
                # Iterate over all facts (assume array structure)
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                main_heatmap = main_heatmaps[heatmap_type][relation_name]
                model_heatmap = model_data[heatmap_type][relation_name]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            elif heatmap_type == "proper_answer_specific_heads":
                # Iterate over all facts and answers
                #for fact_idx in range(len(main_heatmaps[heatmap_type][relation_name])):
                    #for answer_id, main_specific_heatmap in enumerate(
                    #    main_heatmaps[heatmap_type][relation_name][fact_idx]
                    #~):
                main_specific_heatmap = np.logical_or.reduce(list(main_heatmaps[heatmap_type][relation_name].values())) #main_heatmaps[heatmap_type][relation_name][fact_idx]
                model_specific_heatmap = np.logical_or.reduce(list(model_data[heatmap_type][relation_name].values())) #model_data[heatmap_type][relation_name][fact_idx]#[answer_id]

                # Calculate metrics
                intersection = np.logical_and(main_specific_heatmap, model_specific_heatmap)
                union = np.logical_or(main_specific_heatmap, model_specific_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_specific_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

            else:
                # Handle proper_entity_heads and proper_general_heads
                main_heatmap = main_heatmaps[heatmap_type]
                model_heatmap = model_data[heatmap_type]

                # Calculate metrics
                intersection = np.logical_and(main_heatmap, model_heatmap)
                union = np.logical_or(main_heatmap, model_heatmap)

                area_of_intersection = np.sum(intersection)
                area_of_union = np.sum(union)

                iou = area_of_intersection / area_of_union if area_of_union > 0 else 0

                model_metrics[heatmap_type]['iou'].append(iou)
                model_metrics[heatmap_type]['head_count'].append(model_heatmap.sum())
                model_metrics[heatmap_type]['intersection'].append(area_of_intersection)
                model_metrics[heatmap_type]['union'].append(area_of_union)

        # Aggregate results for each heatmap type
        aggregated_metrics = {ht: {
            'iou': np.mean(metrics['iou']),
            'head_count': np.sum(metrics['head_count']),
            'intersection': np.sum(metrics['intersection']),
            'union': np.sum(metrics['union'])
        } for ht, metrics in model_metrics.items()}

        iou_results[model_name] = aggregated_metrics

    return iou_results


def plot_aggregated_proper_heads_iou(iou_results, relation_name, output_dir):
    # Prepare data
    model_names = sorted(
        iou_results.keys(),
        key=lambda name: int(re.search(r'step(\d+)', name).group(1)) if "main" not in name else float('inf')
    )
    general_ious = []
    entity_ious = []
    answer_all_ious = []
    answer_ious = []

    general_head_count = []
    entity_head_count = []
    answer_all_head_count = []
    answer_head_counts = []

    # Populate lists for each heatmap type
    for model_name in model_names[:-1]:
        general_ious.append(iou_results[model_name]['proper_general_heads']['iou'])
        entity_ious.append(iou_results[model_name]['proper_entity_heads']['iou'])
        answer_all_ious.append(iou_results[model_name]['proper_relation_answer_heads']['iou'])
        answer_ious.append(iou_results[model_name]['proper_answer_specific_heads']['iou'])

    for model_name in model_names:
        general_head_count.append(iou_results[model_name]['proper_general_heads']['head_count'])
        entity_head_count.append(iou_results[model_name]['proper_entity_heads']['head_count'])
        answer_all_head_count.append(iou_results[model_name]['proper_relation_answer_heads']['head_count'])
        answer_head_counts.append(iou_results[model_name]['proper_answer_specific_heads']['head_count'])

    # Create directories if not exist
    iou_dir = os.path.join(output_dir, "iou")
    count_dir = os.path.join(output_dir, "count")
    os.makedirs(iou_dir, exist_ok=True)
    os.makedirs(count_dir, exist_ok=True)

    # Plot and save IoU
    plt.figure(figsize=(20, 6))
    plt.plot(model_names[:-1], general_ious, label="Proper General Heads IoU (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names[:-1], entity_ious, label="Proper Entity Heads IoU (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names[:-1], answer_all_ious, label="Proper Relation Answer Heads IoU (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names[:-1], answer_ious, label="Proper Answer Specific Heads IoU (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names[:-1])), labels=model_names[:-1], rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated IoU")
    plt.title(f"Aggregated Proper Heads IoU Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    #plt.savefig(os.path.join(iou_dir, f"{relation_name}_heads_iou.png"))
    plt.close()

    # Plot and save Count
    plt.figure(figsize=(20, 6))
    plt.plot(model_names, general_head_count, label="Proper General Heads Count (Aggregated)", marker='o', linestyle='-')
    plt.plot(model_names, entity_head_count, label="Proper Entity Heads Count (Aggregated)", marker='s', linestyle='--')
    plt.plot(model_names, answer_all_head_count, label="Proper Relation Answer Heads Count (Aggregated)", marker='x', linestyle='-.')
    plt.plot(model_names, answer_head_counts, label="Proper Answer Specific Heads Count (Aggregated)", marker='^', linestyle='-.')

    plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
    plt.xlabel("Model Steps")
    plt.ylabel("Aggregated Count")
    plt.title(f"Aggregated Proper Heads Count Across Models for Relation: {relation_name}")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    #plt.savefig(os.path.join(count_dir, f"{relation_name}_heads_count.png"))
    plt.show()
    plt.close()


# def plot_proper_heads_counts_intersection(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_intersections = []
#     entity_intersections = []
#     answer_all_intersections = []
#     answer_intersections = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_intersections.append(iou_results[model_name]['proper_general_heads']['intersection'])
#         entity_intersections.append(iou_results[model_name]['proper_entity_heads']['intersection'])
#         answer_all_intersections.append(iou_results[model_name]['proper_relation_answer_heads']['intersection'])
#         answer_intersections.append(iou_results[model_name]['proper_answer_specific_heads']['intersection'])

#     # Plotting Intersections
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_intersections, label="Proper General Heads Intersections", marker='o', linestyle='-')
#     plt.plot(model_names, entity_intersections, label="Proper Entity Heads Intersections", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_intersections, label="Proper Relation Answer Heads Intersections", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_intersections, label="Proper Answer Specific Heads Intersections", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Intersection Counts")
#     plt.title(f"Proper Heads Intersections Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


# def plot_proper_heads_counts_union(iou_results, relation_name):
#     # Prepare data
#     model_names = sorted(
#         iou_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )

#     general_unions = []
#     entity_unions = []
#     answer_all_unions = []
#     answer_unions = []

#     # Populate lists for each heatmap type
#     for model_name in model_names:
#         general_unions.append(iou_results[model_name]['proper_general_heads']['union'])
#         entity_unions.append(iou_results[model_name]['proper_entity_heads']['union'])
#         answer_all_unions.append(iou_results[model_name]['proper_relation_answer_heads']['union'])
#         answer_unions.append(iou_results[model_name]['proper_answer_specific_heads']['union'])

#     # Plotting Unions
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_names, general_unions, label="Proper General Heads Unions", marker='o', linestyle='-')
#     plt.plot(model_names, entity_unions, label="Proper Entity Heads Unions", marker='s', linestyle='--')
#     plt.plot(model_names, answer_all_unions, label="Proper Relation Answer Heads Unions", marker='x', linestyle='-.')
#     plt.plot(model_names, answer_unions, label="Proper Answer Specific Heads Unions", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_names)), labels=model_names, rotation=45, ha='right')
#     plt.xlabel("Model Steps")
#     plt.ylabel("Union Counts")
#     plt.title(f"Proper Heads Unions Across Models for Relation: {relation_name}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()

In [ ]:
relation_names = models_data['main'].keys()

In [ ]:
proper_heads['main']['proper_answer_specific_heads']['country_capital_city'][0].sum()

In [ ]:
output_dir = "figures"
for relation_name in relation_names:
    # Extract the sentences for all facts (optional, for reference)
    sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

    # Step 1: Calculate proper heads
    proper_heads = calculate_proper_heads(heatmaps_dict)

    # Step 2: Calculate overlap (IoU) across all facts for the relation
    overlap = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    plot_aggregated_proper_heads_iou(overlap, relation_name, output_dir)

In [ ]:
# overlaps = {}
# for relation_name in relation_names:
#     # Extract the sentences for all facts (optional, for reference)
#     sentences = [models_data['main'][relation_name][fact_idx]['sentence'] for fact_idx in range(len(models_data['main'][relation_name]))]

#     # Step 1: Calculate proper heads
#     proper_heads = calculate_proper_heads(heatmaps_dict)

#     # Step 2: Calculate overlap (IoU) across all facts for the relation
#     overlaps[relation_name] = calculate_consistency_proper_heads_all_facts(proper_heads, relation_name)

    # Step 3: Plot aggregated IoU across all facts for the relation
    #plot_aggregated_proper_heads_iou(overlap, relation_name)

    # # Step 4 (Optional): Add plots for intersections and unions if needed
    # plot_proper_heads_counts_intersection(overlap, relation_name)
    # plot_proper_heads_counts_union(overlap, relation_name)


In [ ]:
# overlaps

## Switches

In [ ]:
# def calculate_switches_proper_heads(heatmaps_dict, relation_name, fact_idx):
#     switch_results = {}

#     # Get the sorted list of model names
#     model_names = list(heatmaps_dict.keys())  # Assume heatmaps_dict is already sorted

#     # Iterate through consecutive model pairs
#     for i in range(len(model_names) - 1):
#         model_name = model_names[i]
#         next_model_name = model_names[i + 1]

#         # Retrieve heatmaps for current and next models
#         current_model_heatmaps = heatmaps_dict[model_name]
#         next_model_heatmaps = heatmaps_dict[next_model_name]

#         # Store switches for this model pair
#         model_switches = {}

#         for heatmap_type in current_model_heatmaps.keys():
#             if heatmap_type == "proper_relation_answer_heads":
#                 # Index by relation_name for relation_answer_heatmaps
#                 current_heatmap = current_model_heatmaps[heatmap_type][relation_name]
#                 next_heatmap = next_model_heatmaps[heatmap_type][relation_name]
#                 switch_map = current_heatmap != next_heatmap
#                 count_map = np.logical_and(current_heatmap, next_heatmap)
#                 count_current = np.sum(current_heatmap)
#                 count_next = np.sum(next_heatmap)
#             elif heatmap_type == "proper_answer_specific_heads":
#                 # Index by relation_name and fact_idx for answer_with_specific
#                 current_heatmap = current_model_heatmaps[heatmap_type][relation_name][fact_idx]
#                 next_heatmap = next_model_heatmaps[heatmap_type][relation_name][fact_idx]
#                 switch_map = current_heatmap != next_heatmap
#                 count_map = np.logical_and(current_heatmap, next_heatmap)
#                 count_current = np.sum(current_heatmap)
#                 count_next = np.sum(next_heatmap)
#             elif heatmap_type == "proper_entity_heads":
#                 current_heatmap = current_model_heatmaps[heatmap_type]
#                 next_heatmap = next_model_heatmaps[heatmap_type]
#                 switch_map = current_heatmap != next_heatmap
#                 count_map = np.logical_and(current_heatmap, next_heatmap)
#                 count_current = np.sum(current_heatmap)
#                 count_next = np.sum(next_heatmap)
#             elif heatmap_type == "proper_general_heads":
#                 current_heatmap = current_model_heatmaps[heatmap_type]
#                 next_heatmap = next_model_heatmaps[heatmap_type]
#                 switch_map = current_heatmap != next_heatmap
#                 count_map = np.logical_and(current_heatmap, next_heatmap)
#                 count_current = np.sum(current_heatmap)
#                 count_next = np.sum(next_heatmap)

#             # Calculate summary statistics, e.g., proportion of switches
#             proportion_switch = np.sum(switch_map) / switch_map.size
#             count_switch = np.sum(switch_map)

#             # Store the switch map and statistics
#             model_switches[heatmap_type] = {
#                 'switch_map': switch_map,
#                 'proportion_switch': proportion_switch,
#                 'count_switch': count_switch,
#                 'count_current': count_current,
#                 'count_next': count_next
#             }

#         # Add this model pair's switch results to the main dictionary
#         switch_results[f"{model_name}"] = model_switches

#     return switch_results


In [ ]:
# def plot_proportion_switches_proper_heads(switch_results, relation_name, fact_idx):
#     # Prepare data
#     model_pairs = sorted(
#         switch_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )
#     general_proportions = []
#     entity_proportions = []
#     answer_all_proportions = []
#     answer_proportions = []

#     # Populate lists for each heatmap type
#     for model_pair in model_pairs:
#         general_proportions.append(switch_results[model_pair]['proper_general_heads']['proportion_switch'])
#         entity_proportions.append(switch_results[model_pair]['proper_entity_heads']['proportion_switch'])
#         answer_all_proportions.append(switch_results[model_pair]['proper_relation_answer_heads']['proportion_switch'])
#         answer_proportions.append(switch_results[model_pair]['proper_answer_specific_heads']['proportion_switch'])

#     # Plotting
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_pairs, general_proportions, label="Proper General Heads", marker='o', linestyle='-')
#     plt.plot(model_pairs, entity_proportions, label="Proper Entity Heads", marker='s', linestyle='--')
#     plt.plot(model_pairs, answer_all_proportions, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
#     plt.plot(model_pairs, answer_proportions, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_pairs)), labels=model_pairs, rotation=45, ha='right')
#     plt.xlabel("Model Pairs")
#     plt.ylabel("Switch Proportion")
#     plt.title(f"Switch Proportion Across Model Pairs for Relation: {relation_name}, Fact: {fact_idx}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


# def plot_count_switches_proper_heads(switch_results, relation_name, fact_idx):
#     # Prepare data
#     model_pairs = sorted(
#         switch_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )
#     general_counts = []
#     entity_counts = []
#     answer_all_counts = []
#     answer_counts = []

#     # Populate lists for each heatmap type
#     for model_pair in model_pairs:
#         general_counts.append(switch_results[model_pair]['proper_general_heads']['count_switch'])
#         entity_counts.append(switch_results[model_pair]['proper_entity_heads']['count_switch'])
#         answer_all_counts.append(switch_results[model_pair]['proper_relation_answer_heads']['count_switch'])
#         answer_counts.append(switch_results[model_pair]['proper_answer_specific_heads']['count_switch'])

#     # Plotting
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_pairs, general_counts, label="Proper General Heads", marker='o', linestyle='-')
#     plt.plot(model_pairs, entity_counts, label="Proper Entity Heads", marker='s', linestyle='--')
#     plt.plot(model_pairs, answer_all_counts, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
#     plt.plot(model_pairs, answer_counts, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_pairs)), labels=model_pairs, rotation=45, ha='right')
#     plt.xlabel("Model Pairs")
#     plt.ylabel("Switch Counts")
#     plt.title(f"Switch Counts Across Model Pairs for Relation: {relation_name}, Fact: {fact_idx}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


# def plot_all_counts_switches_proper_heads(switch_results, relation_name, fact_idx):
#     # Prepare data
#     model_pairs = sorted(
#         switch_results.keys(),
#         key=lambda x: int(x.split('step')[1].split('-')[0])
#     )
    
#     general_counts = []
#     entity_counts = []
#     answer_all_counts = []
#     answer_counts = []

#     # Populate lists for each heatmap type
#     for model_pair in model_pairs:
#         general_counts.append(switch_results[model_pair]['proper_general_heads']['count_current'])
#         entity_counts.append(switch_results[model_pair]['proper_entity_heads']['count_current'])
#         answer_all_counts.append(switch_results[model_pair]['proper_relation_answer_heads']['count_current'])
#         answer_counts.append(switch_results[model_pair]['proper_answer_specific_heads']['count_current'])

#     # Plotting
#     plt.figure(figsize=(20, 6))
#     plt.plot(model_pairs, general_counts, label="Proper General Heads", marker='o', linestyle='-')
#     plt.plot(model_pairs, entity_counts, label="Proper Entity Heads", marker='s', linestyle='--')
#     plt.plot(model_pairs, answer_all_counts, label="Proper Relation Answer Heads", marker='x', linestyle='-.')
#     plt.plot(model_pairs, answer_counts, label="Proper Answer Specific Heads", marker='^', linestyle='-.')

#     plt.xticks(ticks=range(len(model_pairs)), labels=model_pairs, rotation=45, ha='right')
#     plt.xlabel("Model Pairs")
#     plt.ylabel("Total Count")
#     plt.title(f"Total Counts of Specialized Heads Across Model Pairs for Relation: {relation_name}, Fact: {fact_idx}")
#     plt.legend()
#     plt.grid(True)
#     plt.tight_layout()
#     plt.show()


In [ ]:
# relation_name = "country_capital_city"
# fact_idx = 1
# sent = models_data['main'][relation_name][fact_idx]['sentence']
# switches = calculate_switches_proper_heads(proper_heads, relation_name, fact_idx)
# plot_proportion_switches_proper_heads(switches, relation_name, sent)
# plot_count_switches_proper_heads(switches, relation_name, sent)
# #plot_all_counts_switches_proper_heads(switches, relation_name, sent)

In [ ]:
# relation_name = "books_written"
# fact_idx = 1
# sent = models_data['main'][relation_name][fact_idx]['sentence']
# switches = calculate_switches_proper_heads(proper_heads, relation_name, fact_idx)
# plot_proportion_switches_proper_heads(switches, relation_name, sent)
# plot_count_switches_proper_heads(switches, relation_name, sent)
# #plot_all_counts_switches_proper_heads(switches, relation_name, sent)

In [ ]:
# np.sum(proper_heads['step5000-tokens20B']['proper_relation_answer_heads']['books_written'])

### Taking the max activation for multiple token entities and answers

In [ ]:
# heatmaps_dict = {}

# theta = 0.1

# for model_name in sorted_models:
#     relations_data = models_data[model_name]

#     print(model_name)
#     g_total = 0
#     general_heatmap = np.zeros((32, 32), dtype=float)
#     e_total = 0
#     entity_heatmap = np.zeros((32, 32), dtype=float)

#     relation_answer_heatmaps = {}
#     relation_answer_with_specific = {}

#     for relation_name, entries in relations_data.items():

#         relation_answer_heatmap = np.zeros((32, 32), dtype=float)
#         relation_answer_total = 0
#         answer_specific_heatmaps = {}

#         for idx, entry in enumerate(entries):

#             # Ensure subj_token_span is not None
#             if entry["subj_token_span"] is None:
#                 entry["subj_token_span"] = np.arange(0, len(entry["subject_tokens"])).tolist()

#             # Update general heatmap
#             general_heatmap += np.mean(entry['attnheads_contribution_maps'][:entry['answer_token_span'][0]], axis=0) #added [:entry['answer_token_span'][0]] because I dont need the contributions from the answer token itself
#             g_total += 1

#             # Update entity heatmap for subject tokens
#             one_data_map = np.zeros((32, 32), dtype=float)
#             for t in entry["subj_token_span"]:
#                 if t == 0:
#                     e_total -= 1
#                     continue
#                 #one_data_map += entry['attnheads_contribution_maps'][t - 1]
#                 one_data_map = np.maximum(one_data_map, entry['attnheads_contribution_maps'][t - 1])
#             #entity_heatmap += one_data_map
#             entity_heatmap = np.maximum(entity_heatmap, one_data_map)
#             e_total += len(entry["subj_token_span"])

#             # Update entity heatmap and relation answer heatmap for answer tokens
#             one_data_map = np.zeros((32, 32), dtype=float)
#             for t in entry["answer_token_span"]:
#                 one_data_map += entry['attnheads_contribution_maps'][t - 1]
#             entity_heatmap += one_data_map
#             relation_answer_heatmap += one_data_map

#             e_total += len(entry["answer_token_span"])
#             relation_answer_total += len(entry["answer_token_span"])

#             # Store answer-specific heatmap
#             answer_specific_heatmap = one_data_map / len(entry["answer_token_span"])
#             answer_specific_heatmaps[idx] = answer_specific_heatmap  > theta

#         relation_answer_heatmap /= relation_answer_total
#         relation_answer_heatmaps[f"{relation_name}"] = relation_answer_heatmap > theta
#         relation_answer_with_specific[f"{relation_name}"] = answer_specific_heatmaps

#     # Normalize heatmaps
#     general_heatmap /= g_total
#     entity_heatmap /= e_total

#     # Store heatmaps in the dictionary
#     heatmaps_dict[model_name] = {
#         'general_heatmap': general_heatmap > theta,
#         'entity_heatmap': entity_heatmap > theta,
#         'relation_answer_heatmaps': relation_answer_heatmaps,
#         'relation_answer_with_specific': relation_answer_with_specific
#     }

## Head Flow

In [ ]:
# t1 = proper_heads['step5000-tokens20B']
# t2 = proper_heads['step10000-tokens41B']


In [ ]:
# np.logical_and(t1['proper_relation_answer_heads']['country_capital_city'], np.logical_not(t2['proper_relation_answer_heads']['country_capital_city']))

In [ ]:
# t2['proper_general_heads'][t1['proper_relation_answer_heads']['country_capital_city'] != t2['proper_relation_answer_heads']['country_capital_city']]

In [ ]:
# t1_relation_heads = t1['proper_relation_answer_heads']['country_capital_city']
# t2_relation_heads = t2['proper_relation_answer_heads']['country_capital_city']
# t2_general_heads = t2['proper_general_heads']
# t2_proper_entity_heads = t2['proper_entity_heads']

# # Step 1: Identify indices where `t1` is True but `t2` is False
# true_to_false_indices = np.logical_and(t1_relation_heads, np.logical_not(t2_relation_heads))

# # Step 2: Check if these indices are True in the general heads map of the second revision
# true_in_general_heads = t2_general_heads[true_to_false_indices]
# true_in_entity_heads = t2_proper_entity_heads[true_to_false_indices]


In [ ]:
#proper_heads['main'].keys()

In [ ]:
# def aggregate_heads(map_data):
#     """Aggregate facts or relations using np.logical_or."""
#     return np.logical_or.reduce(list(map_data.values()))

# #relation_names = models_data['main'].keys()

# results = {map_type: {} for map_type in map_types}

# for map_type in map_types:
#     for i in range(len(revisions) - 1):
#         rev1, rev2 = revisions[i], revisions[i + 1]
#         data1, data2 = proper_heads[rev1][map_type], proper_heads[rev2][map_type]

#         # Handle aggregated maps for `proper_relation_answer_heads` and `proper_answer_specific_heads`
#         if map_type == "proper_answer_specific_heads":
#             data1 = aggregate_heads(data1)
#             data2 = aggregate_heads(data2)

#         # Step 3: Detect changes
#         true_to_false = np.logical_and(data1, np.logical_not(data2))
#         false_to_true = np.logical_and(np.logical_not(data1), data2)

#         # Step 4: Track changes across other map types (optional)
#         changes_in_general = proper_heads[rev2]["proper_general_heads"][true_to_false | false_to_true]
#         changes_in_entity = proper_heads[rev2]["proper_entity_heads"][true_to_false | false_to_true]

#         # Initialize per-relation data storage
#         relation_changes = {}

#         for rel in relation_names:
#             # Detect changes in relation-specific maps
#             changes_in_relation_answer = proper_heads[rev2]["proper_relation_answer_heads"][rel][true_to_false | false_to_true]
#             changes_in_answer_specific = aggregate_heads(proper_heads[rev2]["proper_answer_specific_heads"][rel])[true_to_false | false_to_true]

#             # Store results per relation
#             relation_changes[rel] = {
#                 "changes_in_relation_answer": np.sum(changes_in_relation_answer),
#                 "changes_in_answer_specific": np.sum(changes_in_answer_specific),
#             }

#         # Step 5: Store overall results
#         results[map_type][f"{rev1} → {rev2}"] = {
#             "true_to_false_count": np.sum(true_to_false),
#             "false_to_true_count": np.sum(false_to_true),
#             "total_changes": np.sum(true_to_false | false_to_true),
#             "changes_in_general_heads": np.sum(changes_in_general),
#             "changes_in_entity_heads": np.sum(changes_in_entity),
#             "relation_changes": relation_changes,  # Add per-relation changes here
#         }

# # Step 6: Print results
# for map_type, map_results in results.items():
#     print(f"Results for {map_type}:")
#     for change, data in map_results.items():
#         print(f"  {change}: {data}")
#         print(f"    True to False Count: {data['true_to_false_count']}")
#         print(f"    False to True Count: {data['false_to_true_count']}")
#         print(f"    Total Changes: {data['total_changes']}")
#         print(f"    Changes in General Heads: {data['changes_in_general_heads']}")
#         print(f"    Changes in Entity Heads: {data['changes_in_entity_heads']}")
#         print(f"    Relation-Specific Changes:")
#         for rel, rel_data in data["relation_changes"].items():
#             print(f"      Relation: {rel}")
#             print(f"        Changes in Relation Answer: {rel_data['changes_in_relation_answer']}")
#             print(f"        Changes in Answer Specific: {rel_data['changes_in_answer_specific']}")

In [ ]:
map_types

### Head Contribution Flow on a Relation Level

In [ ]:
# results = {map_type: {} for map_type in map_types}

# for map_type in map_types:
#     for i in range(len(revisions) - 1):
#         rev1, rev2 = revisions[i], revisions[i + 1]
#         data1 = proper_heads[rev1][map_type]
#         data2 = proper_heads[rev2][map_type]

#         if map_type == "proper_answer_specific_heads":
#             # Reduce facts to relations and then aggregate
#             data1_agg = np.logical_or.reduce(
#                     [np.logical_or.reduce(list(fact_maps.values())) for fact_maps in list(data1.values())]
#                 )
#             total_heads_rev1 = np.sum(data1_agg)
            
#             data2_agg = np.logical_or.reduce(
#                     [np.logical_or.reduce(list(fact_maps.values())) for fact_maps in list(data2.values())]
#                 )
#             total_heads_rev2 = np.sum(data2_agg)
    
#         elif map_type == "proper_relation_answer_heads":
#             # Aggregate relations directly
#             data1_agg = np.logical_or.reduce(list(data1.values()))
#             total_heads_rev1 = np.sum(data1_agg)
#             data2_agg = np.logical_or.reduce(list(data2.values()))
#             total_heads_rev2 = np.sum(data2_agg)
#         else:
#             # Directly use the non-relation-specific data
#             data1_agg = data1
#             total_heads_rev1 = np.sum(data1_agg)
            
#             data2_agg = data2
#             total_heads_rev2 = np.sum(data2_agg)

#         # Step 1: Detect true-to-false and false-to-true transitions
#         true_to_false = np.logical_and(data1_agg, np.logical_not(data2_agg))
#         false_to_true = np.logical_and(np.logical_not(data1_agg), data2_agg)

#         # Step 2: Track role switches
#         switched_into_other_roles = np.zeros_like(true_to_false, dtype=bool)
#         stemming_from_other_roles = np.zeros_like(false_to_true, dtype=bool)
#         switched_roles = {}

#         for other_type in map_types:
#             if other_type != map_type:
#                 other_data1 = proper_heads[rev1][other_type]
#                 other_data2 = proper_heads[rev2][other_type]

#                 # Handle relation-specific maps
#                 if isinstance(other_data1, dict):  # If the data is relation-specific
#                     if other_type == "proper_answer_specific_heads":
#                         # Reduce facts to relations using logical OR
#                         other_data1_agg = {
#                             rel: np.logical_or.reduce(fact_maps)
#                             for rel, fact_maps in other_data1.items()
#                         }
#                         other_data2_agg = {
#                             rel: np.logical_or.reduce(fact_maps)
#                             for rel, fact_maps in other_data2.items()
#                         }
#                     else:  # For proper_relation_answer_heads, keep maps per relation
#                         other_data1_agg = other_data1  # Keep per relation
#                         other_data2_agg = other_data2
#                 else:
#                     # For non-relation-specific maps, directly use the data
#                     other_data1_agg = other_data1
#                     other_data2_agg = other_data2

#                 # Compute role switches and contributions
#                 if isinstance(other_data1_agg, dict):  # For relation-specific data
#                     for rel in other_data1_agg:
#                         switched_into_other_roles = np.logical_or(
#                             switched_into_other_roles,
#                             np.logical_and(true_to_false, other_data2_agg[rel])
#                         )
#                         # stemming_from_other_roles = np.logical_or(
#                         #     stemming_from_other_roles,
#                         #     np.logical_and(false_to_true, other_data1_agg[rel])
#                         # )
#                         switched = np.logical_and(np.logical_not(other_data1_agg[rel]), other_data2_agg[rel])
#                         switched_roles[rel] = switched_roles.get(rel, 0) + np.sum(switched)
#                 else:
#                     switched_into_other_roles = np.logical_or(
#                         switched_into_other_roles,
#                         np.logical_and(true_to_false, other_data2_agg)
#                     )
#                     # stemming_from_other_roles = np.logical_or(
#                     #     stemming_from_other_roles,
#                     #     np.logical_and(false_to_true, other_data1_agg)
#                     # )
#                     switched = np.logical_and(np.logical_not(other_data1_agg), other_data2_agg)
#                     switched_roles[other_type] = np.sum(switched)
                    

#                 # Step 3: Final deactivated and new heads
#                 deactivated_heads = np.logical_and(true_to_false, np.logical_not(switched_into_other_roles))
#                 deactivated_count = np.sum(deactivated_heads)

#                 #new_heads = np.logical_and(false_to_true, np.logical_not(stemming_from_other_roles))
#                 new_heads_count = np.sum(false_to_true)

#                 # Store results for this map type and revision pair
#                 results[map_type][f"{rev1} → {rev2}"] = {
#                     "total_heads_rev1": total_heads_rev1,
#                     "total_heads_rev2": total_heads_rev2,
#                     "deactivated_heads": deactivated_count,
#                     "changed": np.sum(true_to_false),
#                     "new_heads": new_heads_count,
#                     "switched_roles": switched_roles,
#                     }

In [ ]:
# results = {map_type: {} for map_type in map_types}

# for map_type in map_types:
#     for i in range(len(revisions) - 1):
#         rev1, rev2 = revisions[i], revisions[i + 1]

#         # Retrieve and aggregate data
#         data1 = proper_heads[rev1][map_type]
#         data2 = proper_heads[rev2][map_type]
#         data1_agg, data1_per_relation = aggregate_heads(data1, map_type)
#         data2_agg, data2_per_relation = aggregate_heads(data2, map_type)

#         # Calculate total active heads for each revision
#         total_heads_rev1 = np.sum(data1_agg)
#         total_heads_rev2 = np.sum(data2_agg)

#         # Initialize counts for general and entity heads
#         general_head_count = np.sum(proper_heads[rev1]["proper_general_heads"])
#         entity_head_count = np.sum(proper_heads[rev1]["proper_entity_heads"])

#         # Step 1: Detect true-to-false and false-to-true transitions
#         true_to_false = np.logical_and(data1_agg, np.logical_not(data2_agg))
#         false_to_true = np.logical_and(np.logical_not(data1_agg), data2_agg)

#         # Step 2: Track role switches
#         switched_into_other_roles = np.zeros_like(true_to_false, dtype=bool)
#         switched_roles = {}  # Track per relation if available
#         overall_switched_roles = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}

#         for other_type in map_types:
#             if other_type != map_type:
#                 other_data1 = proper_heads[rev1][other_type]
#                 other_data2 = proper_heads[rev2][other_type]

#                 # Aggregate other data
#                 other_data1_agg, other_data1_per_relation = aggregate_heads(other_data1, other_type)
#                 other_data2_agg, other_data2_per_relation = aggregate_heads(other_data2, other_type)

#                 if other_data1_per_relation and data1_per_relation:  # If relation-specific, iterate per relation
#                     for rel in data1_per_relation:
#                         rel_true_to_false = np.logical_and(
#                             data1_per_relation[rel], np.logical_not(data2_per_relation.get(rel, np.zeros_like(data1_per_relation[rel])))
#                         )
#                         rel_switched_into_other_roles = np.logical_and(
#                             rel_true_to_false, other_data2_per_relation.get(rel, np.zeros_like(rel_true_to_false))
#                         )
#                         switched_roles.setdefault(rel, {}).update({
#                             other_type: np.sum(rel_switched_into_other_roles)
#                         })

#                 switched_into_other_roles = np.logical_or(
#                     switched_into_other_roles,
#                     np.logical_and(true_to_false, other_data2_agg)
#                 )

#                 # Track switches for overall role counts
#                 role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
#                 if other_type == "proper_general_heads":
#                     overall_switched_roles["general"] += np.sum(role_switch_contribution)
#                 elif other_type == "proper_entity_heads":
#                     overall_switched_roles["entity"] += np.sum(role_switch_contribution)
#                 elif other_type == "proper_relation_answer_heads":
#                     overall_switched_roles["relation_answer"] += np.sum(role_switch_contribution)
#                 elif other_type == "proper_answer_specific_heads":
#                     overall_switched_roles["answer_specific"] += np.sum(role_switch_contribution)

#         # Step 3: Final deactivated and new heads
#         deactivated_heads = np.logical_and(true_to_false, np.logical_not(switched_into_other_roles))
#         deactivated_count = np.sum(deactivated_heads)

#         new_heads = np.logical_and(false_to_true, np.logical_not(switched_into_other_roles))
#         new_heads_count = np.sum(new_heads)

#         # Store results for this map type and revision pair
#         results[map_type][f"{rev1} → {rev2}"] = {
#             "total_heads_rev1": total_heads_rev1,
#             "total_heads_rev2": total_heads_rev2,
#             "general_head_count": general_head_count,
#             "entity_head_count": entity_head_count,
#             "deactivated_heads": deactivated_count,
#             "changed": np.sum(true_to_false),
#             "new_heads": new_heads_count,
#             "overall_switched_roles": overall_switched_roles,
#             "per_relation_switched_roles": switched_roles,
#         }

# # Print results
# for map_type, map_results in results.items():
#     print(f"Results for {map_type}:")
#     for change, data in map_results.items():
#         print(f"  {change}:")
#         print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
#         print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")
#         print(f"    General Head Count: {data['general_head_count']}")
#         print(f"    Entity Head Count: {data['entity_head_count']}")
#         print(f"    Deactivated Heads: {data['deactivated_heads']}")
#         print(f"    Changed Heads: {data['changed']}")
#         print(f"    New Heads: {data['new_heads']}")
#         print(f"    Overall Role Switches: {data['overall_switched_roles']}")
#         print(f"    Per-Relation Role Switches:")
#         for rel, switches in data["per_relation_switched_roles"].items():
#             print(f"      Relation: {rel}")
#             for other_type, count in switches.items():
#                 print(f"        {other_type}: {count}")


In [ ]:
# # Print aggregated results for each map type
# for map_type, map_results in results.items():
#     print(f"Results for {map_type}:")
#     for change, data in map_results.items():
#         print(f"  {change}:")  # Example: "step5000-tokens20B → step10000-tokens41B"

#         # Print total heads for each revision
#         print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
#         print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")

#         # Print deactivated and new heads
#         print(f"    Deactivated Heads: {data['deactivated_heads']}")
#         print(f"    Changed: {data['changed']}")
#         print(f"    New Heads: {data['new_heads']}")

#         # Print role switches
#         print(f"    Role Switches:")
#         for other_type, count in data["switched_roles"].items():
#             if count > 0:
#                 print(f"      {map_type} → {other_type}: {count}")

### Head Contribution averaged Relation
Here it is also possible that some heads are active in relation_answer and answer_specific since we averaged over all the realtion and therefore lose the property of proper heads \
This is one output: \
Results for proper_general_heads:\
  step5000-tokens20B → step10000-tokens41B:\
    Total Heads (Rev1): 94\
    Total Heads (Rev2): 107\
    Deactivated Heads: 0\
    Changed Heads: 2\
    New Heads: 15\
    Role Switches: {'general': 0, 'entity': 0, 'relation_answer': 2, 'answer_specific': 1}\

    So from the first to the second revision step 15 heads are getting activated and two are getting changed. The difference between changed and deactivated is that the changed ones change their job. The deactivated one wont get used in the next step. General or Entity plus the answer ones cant be more than the changed heads
    

In [ ]:
def aggregate_answer_specific_heads(data):
    """
    Aggregate proper_answer_specific_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(
        [np.logical_or.reduce(list(fact_maps.values())) for fact_maps in list(data.values())]
    )

def aggregate_relation_answer_heads(data):
    """
    Aggregate proper_relation_answer_heads into a single 32x32 map.
    """
    return np.logical_or.reduce(list(data.values()))

revisions = sorted_models

map_types = ["proper_general_heads", "proper_entity_heads", 
             "proper_relation_answer_heads", "proper_answer_specific_heads"]

proper_heads = calculate_proper_heads(heatmaps_dict)

In [ ]:
# # Print aggregated results for each map type
# for map_type, map_results in results.items():
#     print(f"Results for {map_type}:")
#     for change, data in map_results.items():
#         print(f"  {change}:")  # Example: "step5000-tokens20B → step10000-tokens41B"

#         # Print total heads for each revision
#         print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
#         print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")

#         # Print deactivated and new heads
#         print(f"    Deactivated Heads: {data['deactivated_heads']}")
#         print(f"    Changed: {data['changed']}")
#         print(f"    New Heads: {data['new_heads']}")

#         # Print role switches
#         print(f"    Role Switches:")
#         for other_type, count in data["switched_roles"].items():
#             if count > 0:
#                 print(f"      {map_type} → {other_type}: {count}")

In [ ]:
revisions_2 = ['step5000-tokens20B', 'step50000-tokens209B', 'step100000-tokens419B','step200000-tokens838B', 'main']

In [ ]:
revisions

In [ ]:
proper_heads

In [ ]:
def logical_or_recursive(data):
    """
    Recursively applies logical OR across all boolean arrays in a nested dictionary structure.
    
    Args:
    - data (dict): Nested dictionary with numpy boolean arrays at various levels.
    
    Returns:
    - np.array: A numpy array that is the result of applying logical OR across all found boolean arrays.
    """
    result = None

    if isinstance(data, dict):
        for key, value in data.items():
            sub_result = logical_or_recursive(value)  # Recursive call
            if sub_result is not None:
                if result is None:
                    result = sub_result
                else:
                    result |= sub_result  # Element-wise logical OR
    elif isinstance(data, np.ndarray) and data.dtype == bool:
        return data  # Base case: return boolean array

    return result

t1 = logical_or_recursive(proper_heads)

In [ ]:
def recursive_logical_or(data, result=None):
    for key, value in data.items():
        if isinstance(value, dict):  # If value is a dictionary, recurse
            result = recursive_logical_or(value, result)
        else:  # If value is an array, apply logical OR
            array = np.array(value, dtype=bool)
            print(array.sum())
            if result is None:
                result = array.copy()
            else:
                result |= array  # Logical OR operation
    return result

# Apply recursive logical OR operation
final_result = recursive_logical_or(proper_heads)

In [ ]:
final_result.sum()

In [ ]:
results = {map_type: {} for map_type in map_types}

for map_type in map_types:
    for i in range(len(revisions) - 1):
        rev1, rev2 = revisions[i], revisions[i + 1]

        # Get the data for the current map_type
        data1 = proper_heads[rev1][map_type]
        data2 = proper_heads[rev2][map_type]

        # Aggregate maps if necessary
        if map_type == "proper_answer_specific_heads":
            data1_agg = aggregate_answer_specific_heads(data1)
            data2_agg = aggregate_answer_specific_heads(data2)
        elif map_type == "proper_relation_answer_heads":
            data1_agg = aggregate_relation_answer_heads(data1)
            data2_agg = aggregate_relation_answer_heads(data2)
        else:
            data1_agg = data1
            data2_agg = data2

        # Step 1: Detect true-to-false and false-to-true transitions
        true_to_false = np.logical_and(data1_agg, np.logical_not(data2_agg))
        false_to_true = np.logical_and(np.logical_not(data1_agg), data2_agg)

        # Step 2: Track role switches
        switched_into_other_roles = np.zeros_like(true_to_false, dtype=bool)
        stemming_from_other_roles = np.zeros_like(false_to_true, dtype=bool)

        role_switch_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}
        role_stemming_counts = {"general": 0, "entity": 0, "relation_answer": 0, "answer_specific": 0}

        switched_indices = []
        stemmed_indices = []

        for other_type in map_types:
            if other_type != map_type:
                other_data1 = proper_heads[rev1][other_type]
                other_data2 = proper_heads[rev2][other_type]

                # Aggregate if relation-specific
                #if isinstance(other_data1, dict):
                if other_type == "proper_answer_specific_heads":
                    other_data1_agg = aggregate_answer_specific_heads(other_data1)
                    other_data2_agg = aggregate_answer_specific_heads(other_data2)
                elif other_type == "proper_relation_answer_heads":
                    other_data1_agg = aggregate_relation_answer_heads(other_data1)
                    other_data2_agg = aggregate_relation_answer_heads(other_data2)
                else:
                    # For global maps, use directly
                    other_data1_agg = other_data1
                    other_data2_agg = other_data2

                
                switched_into_other_roles = np.logical_or(
                    switched_into_other_roles,
                    np.logical_and(true_to_false, other_data2_agg)
                )
                
                stemming_from_other_roles = np.logical_or(
                        stemming_from_other_roles,
                        np.logical_and(false_to_true, other_data1_agg)
                )
                
                switched_mask = np.logical_and(true_to_false, other_data2_agg)
                stemmed_mask = np.logical_and(false_to_true, other_data1_agg)
                
                switched_indices.extend(list(zip(*np.where(switched_mask))))
                stemmed_indices.extend(list(zip(*np.where(stemmed_mask))))

                # Contribution counts
                role_switch_contribution = np.sum(switched_mask)
                role_stemming_contribution = np.sum(stemmed_mask)
                
                role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
                role_switch_contribution_stemming = np.logical_and(false_to_true, other_data1_agg)

                if other_type == "proper_general_heads":
                    role_switch_counts["general"] += np.sum(role_switch_contribution)
                    role_stemming_counts["general"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_entity_heads":
                    role_switch_counts["entity"] += np.sum(role_switch_contribution)
                    role_stemming_counts["entity"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_relation_answer_heads":
                    role_switch_counts["relation_answer"] += np.sum(role_switch_contribution)
                    role_stemming_counts["relation_answer"] += np.sum(role_switch_contribution_stemming)
                elif other_type == "proper_answer_specific_heads":
                    role_switch_counts["answer_specific"] += np.sum(role_switch_contribution)
                    role_stemming_counts["answer_specific"] += np.sum(role_switch_contribution_stemming)
                # role_switch_contribution = np.logical_and(true_to_false, other_data2_agg)
                # # Aggregate switches into categories
                # #switched = np.logical_and(np.logical_not(other_data1_agg), other_data2_agg)
                # if other_type == "proper_general_heads":
                #     role_switch_counts["general"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_entity_heads":
                #     role_switch_counts["entity"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_relation_answer_heads":
                #     role_switch_counts["relation_answer"] += np.sum(role_switch_contribution)
                # elif other_type == "proper_answer_specific_heads":
                #     role_switch_counts["answer_specific"] += np.sum(role_switch_contribution)

        # Step 3: Final deactivated and new heads
        deactivated = np.logical_and(true_to_false, np.logical_not(switched_into_other_roles))
        deactivated_count = np.sum(deactivated)

        new_heads = np.logical_and(false_to_true, np.logical_not(stemming_from_other_roles))
        new_heads_count = np.sum(new_heads)

        # Store results for this map type and revision pair
        results[map_type][f"{rev1} → {rev2}"] = {
            "total_heads_rev1": np.sum(data1_agg),
            "total_heads_rev2": np.sum(data2_agg),
            "true_to_false": np.sum(true_to_false),#deactivated_count,
            "deactivated_count": deactivated_count, #np.sum(true_to_false),  # Changed heads
            "new_heads": new_heads_count,
            "role_switch_counts": role_switch_counts,  # Aggregated role switches
            "role_stemming_counts": role_stemming_counts,
            "switched_indices": switched_indices,  # Indices of switched heads
            "stemmed_indices": stemmed_indices,
        }

In [ ]:
data = results.copy()

In [ ]:
for map_type, map_results in results.items():
    print(f"Results for {map_type}:")
    for change, data in map_results.items():
        print(f"  {change}:")
        print(f"    Total Heads (Rev1): {data['total_heads_rev1']}")
        print(f"    Total Heads (Rev2): {data['total_heads_rev2']}")
        print(f"    True to False: {data['true_to_false']}")
        print(f"    Deactivated Heads: {data['deactivated_count']}")
        print(f"    New Heads: {data['new_heads']}")
        print(f"    Role Switches: {data['role_switch_counts']}")
        print(f"    Role Stemming: {data['role_stemming_counts']}")
        print(f"    Switched heads in {map_type}: {data['switched_indices']}")
        print(f"    Stemmed heads in {map_type}: {data['stemmed_indices']}")

In [ ]:
results

In [ ]:
from collections import Counter

# Initialize counters for layers
switched_layer_counts = Counter()
switched_head_counts = Counter()

# Iterate through results to collect switched and stemmed head layer counts
for map_type in results:
    for transition in results[map_type]:
        # Get switched and stemmed indices
        switched_heads = results[map_type][transition]["switched_indices"]

        # Count occurrences for each layer and (layer, head) pair in switched heads
        for layer, head in switched_heads:
            switched_layer_counts[layer] += 1
            switched_head_counts[(layer, head)] += 1
            
# Print switched head layer frequencies
print("Layer change frequencies (Switched Heads):")
for layer, count in sorted(switched_layer_counts.items()):
    print(f"Layer {layer}: {count} times")

# Print switched (layer, head) frequencies
print("\nHead change frequencies (Switched Heads):")
for (layer, head), count in sorted(switched_head_counts.items()):
    print(f"Layer {layer}, Head {head}: {count} times")

In [ ]:
results

In [ ]:
total_heads_per_revision = {}

for map_type in map_types:
    for i in range(len(revisions)):
        rev1 = revisions[i]
        data1 = proper_heads[rev1][map_type]
        # Get the data for the current map_type
        if map_type == "proper_relation_answer_heads":
            data2 = proper_heads[rev1]["proper_answer_specific_heads"]
            data1_agg_ra = aggregate_relation_answer_heads(data1)
            data2_agg_as = aggregate_answer_specific_heads(data2)
            
            data1_agg = np.logical_or(data1_agg_ra, data2_agg_as)
        else:
            # For other types, use the data directly or aggregate if necessary
            if map_type == "proper_answer_specific_heads":
                data1_agg = aggregate_answer_specific_heads(data1)
            else:
                data1_agg = data1
        
        
        if revisions[i] not in total_heads_per_revision:
            total_heads_per_revision[revisions[i]] = {
                "total_count": 0,
                "intersection_count": 0,
                "proper_general_heads": 0,
                "proper_entity_heads": 0,
                "proper_relation_answer_heads": 0,
                "proper_answer_specific_heads": 0,
            }
        if map_type == "proper_relation_answer_heads":
            # Add total count to revision and specific map type
            total_heads_per_revision[revisions[i]]["total_count"] += np.sum(data1_agg)
            total_heads_per_revision[revisions[i]]["intersection_count"] += np.sum(data1_agg)
            total_heads_per_revision[revisions[i]][map_type] += np.sum(data1_agg_ra)
        elif map_type == "proper_answer_specific_heads":
            total_heads_per_revision[revisions[i]][map_type] += np.sum(data1_agg)
        else:
            total_heads_per_revision[revisions[i]]["total_count"] += np.sum(data1_agg)
            total_heads_per_revision[revisions[i]][map_type] += np.sum(data1_agg)

df_total_heads_revisions = pd.DataFrame.from_dict(total_heads_per_revision, orient="index")
df_total_heads_revisions

In [ ]:
total_heads_per_revision = {}

for map_type in map_types:
    for i in range(len(revisions)):
        rev1 = revisions[i]
        # Get the data for the current map_type
        data1 = proper_heads[rev1][map_type]

        # Aggregate maps if necessary
        if map_type == "proper_answer_specific_heads":
            data1_agg = aggregate_answer_specific_heads(data1)
        elif map_type == "proper_relation_answer_heads":
            data1_agg = aggregate_relation_answer_heads(data1)
        else:
            data1_agg = data1
        
        
        if revisions[i] not in total_heads_per_revision:
            total_heads_per_revision[revisions[i]] = {
                "total_count": 0,
                "proper_general_heads": 0,
                "proper_entity_heads": 0,
                "proper_relation_answer_heads": 0,
                "proper_answer_specific_heads": 0,
            }

        # Add total count to revision and specific map type
        total_heads_per_revision[revisions[i]]["total_count"] += np.sum(data1_agg)
        total_heads_per_revision[revisions[i]][map_type] += np.sum(data1_agg)

df_total_heads_revisions = pd.DataFrame.from_dict(total_heads_per_revision, orient="index")
df_total_heads_revisions

In [ ]:
import matplotlib.pyplot as plt

# Extracting data for plotting
x_values = df_total_heads_revisions.index  # Step revisions
y_values = df_total_heads_revisions[["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads", "total_count"]]

# Plot the data
plt.figure(figsize=(12, 6))
for column in y_values.columns:
    plt.plot(x_values, y_values[column], marker='o', linestyle='-', label=column)

# Formatting the plot
plt.xlabel("Model Revision (Step)")
plt.ylabel("Count")
plt.title("Evolution of Head Types Across Model Revisions")
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.legend()
plt.grid(True)

# Show the plot
plt.show()


In [ ]:
ratios_matrices = {}

# Mapping from extracted_matrix row names to df_row names
mapping = {
    "General Heads": "proper_general_heads",
    "Entity Heads": "proper_entity_heads",
    "Relation Answer Heads": "proper_relation_answer_heads",
    "Answer Specific Heads": "proper_answer_specific_heads",
}

for transition, extracted_matrix in extracted_matrices.items():
    # Extract the source step before transition
    source_step = transition.split(" → ")[0]
    
    # Get the corresponding row from df_total_heads_revisions
    df_row = df_total_heads_revisions.loc[source_step]
    
    # Create a new matrix to store the ratios
    ratio_matrix = extracted_matrix.copy()
    
    for category in extracted_matrix.index:  # Iterate over row names
        if category in mapping:  # Ensure it's a known category
            denominator = df_row.get(mapping[category], 1)  # Use the correct total count for this head type (default to 1 to avoid division by zero)
        elif category == "Deactivated Count":
            denominator = df_row.get("total_count", 1)  # Use the overall total count for deactivated heads
        else:
            denominator = 1  # Default to 1 to avoid errors

        for col in extracted_matrix.columns:  # Iterate over columns
            if denominator != 0:
                ratio_matrix.loc[category, col] = extracted_matrix.loc[category, col] / denominator
            else:
                ratio_matrix.loc[category, col] = 0  # Set to 0 if denominator is 0

    # Store the ratio matrix
    ratios_matrices[transition] = ratio_matrix


In [ ]:
switched_head_counts

In [ ]:
df = pd.DataFrame([(layer, head, count) for (layer, head), count in switched_head_counts.items()],
                  columns=["Layer", "Head", "Switches"])

# Plot
plt.figure(figsize=(12, 8))
plt.scatter(df["Head"], df["Layer"], s=df["Switches"] * 10, alpha=0.6, edgecolors="black")

plt.xlabel("Head Number")
plt.ylabel("Layer Number")
plt.title("Head Change Frequencies (Switched Heads)")

plt.xticks(range(0, 32, 2))
plt.yticks(range(df["Layer"].min(), df["Layer"].max() + 1))

plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [ ]:
# Define the correct order of map types for subplots
map_order = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Initialize a dictionary to store counters per map_type
switched_head_counts_per_map = {map_type: Counter() for map_type in map_order}

# Iterate through results to collect switched head counts per map_type
for map_type in results:
    if map_type in switched_head_counts_per_map:  # Ensure we only process expected types
        for transition in results[map_type]:
            switched_heads = results[map_type][transition]["switched_indices"]
            for layer, head in switched_heads:
                switched_head_counts_per_map[map_type][(layer, head)] += 1

# Convert to DataFrame format for plotting
plot_data = []
for map_type in map_order:
    counter = switched_head_counts_per_map[map_type]
    for (layer, head), count in counter.items():
        plot_data.append((map_type, layer, head, count))

df = pd.DataFrame(plot_data, columns=["Map Type", "Layer", "Head", "Switches"])

# Plot subplots in the specified order
num_maps = len(map_order)
fig, axes = plt.subplots(1, num_maps, figsize=(6 * num_maps, 6), sharey=True)

for ax, map_type in zip(axes, map_order):
    data = df[df["Map Type"] == map_type]
    if not data.empty:
        ax.scatter(data["Head"], data["Layer"], s=data["Switches"] * 10, alpha=0.6, edgecolors="black")
        ax.set_title(f"Switched Heads - {map_type}")
        ax.set_xlabel("Head Number")
        ax.set_ylabel("Layer Number")
        ax.set_xticks(range(0, 32, 2))
        ax.set_yticks(range(df["Layer"].min(), df["Layer"].max() + 1))
        ax.grid(True, linestyle="--", alpha=0.5)
    else:
        ax.set_visible(False)  # Hide subplot if there's no data

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from collections import Counter

# Define the correct order of map types for subplots
map_order = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Initialize a dictionary to store counters per map_type
switched_head_counts_per_map = {map_type: Counter() for map_type in map_order}

# Iterate through results to collect switched head counts per map_type
for map_type in results:
    if map_type in switched_head_counts_per_map:  # Ensure we only process expected types
        for transition in results[map_type]:
            switched_heads = results[map_type][transition]["switched_indices"]
            for layer, head in switched_heads:
                switched_head_counts_per_map[map_type][(layer, head)] += 1

# Convert to DataFrame format for plotting
plot_data = []
for map_type in map_order:
    counter = switched_head_counts_per_map[map_type]
    for (layer, head), count in counter.items():
        plot_data.append((map_type, layer, head, count))

df = pd.DataFrame(plot_data, columns=["Map Type", "Layer", "Head", "Switches"])

# Create a 2x2 grid for subplots
fig, axes = plt.subplots(2, 2, figsize=(12, 12), sharey=True, sharex=True)

# Flatten the 2D array of axes for easier iteration
axes = axes.flatten()

for ax, map_type in zip(axes, map_order):
    data = df[df["Map Type"] == map_type]
    if not data.empty:
        ax.scatter(data["Head"], data["Layer"], s=data["Switches"] * 10, alpha=0.6, edgecolors="black")
        ax.set_title(f"Switched Heads - {map_type}")
        ax.set_xlabel("Head Number")
        ax.set_ylabel("Layer Number")
        ax.set_xticks(range(0, 32, 2))
        ax.set_yticks(range(df["Layer"].min(), df["Layer"].max() + 1))
        ax.grid(True, linestyle="--", alpha=0.5)
    else:
        ax.set_visible(False)  # Hide subplot if there's no data

plt.tight_layout()
plt.show()

In [ ]:
# Define the transitions to extract
transitions = []
category_keys = ["proper_general_heads", "proper_entity_heads", "proper_relation_answer_heads", "proper_answer_specific_heads"]

# Define the value keys (column names)
value_keys = ["general", "entity", "relation_answer", "answer_specific", "deactivated_count"]

for category in results.values():
    transitions =list(category.keys())
    break# Collect all unique transitions

# Convert transitions to a sorted list for consistency
#transitions = set(transitions)#sorted(list(transitions))
# Dictionary to store the extracted matrices
extracted_matrices = {}

# Iterate over transitions and extract data
for transition in transitions:
    extracted_matrix = np.zeros((len(category_keys) + 1, len(value_keys)))

    # Populate the matrix with correct values
    for i, category in enumerate(category_keys):  # Last category is for deactivated_count
        total_counts = {key: 0 for key in value_keys}  # Default values
        if transition in results.get(category, {}):
            total_counts.update(results[category][transition].get("role_switch_counts", {}))
            total_counts["deactivated_count"] += results[category][transition].get("deactivated_count", 0)
        extracted_matrix[i] = [total_counts[v] for v in value_keys]

    # Manually populate the last row with deactivated counts
    #extracted_matrix[-1] = [0, 0, 0, 0, sum(results[cat][transition].get("deactivated_count", 0) for cat in results.keys() if transition in results[cat])]
    new_heads_counts = {key: 0 for key in value_keys}
    for i, category in enumerate(category_keys[:-1]):  # Map new heads to the respective categories
        if transition in results.get(category, {}):
            new_heads_counts[value_keys[i]] += results[category][transition].get("new_heads", 0)

    extracted_matrix[-1] = [new_heads_counts[v] for v in value_keys]  # Assign correctly

    # Convert to DataFrame and store
    df_extracted = pd.DataFrame(
        extracted_matrix,
        index=["General Heads", "Entity Heads", "Relation Answer Heads", "Answer Specific Heads", "Deactivated Count"],
        columns=value_keys
    )
    
    extracted_matrices[transition] = df_extracted

    # Display each extracted matrix
    #tools.display_dataframe_to_user(name=f"Extracted Matrix: {transition}", dataframe=df_extracted)


In [ ]:
extracted_matrices.items()

In [ ]:
for i, (transition, extracted_matrix) in enumerate(extracted_matrices.items()):
    print(i)
    print(transition)
    print(extracted_matrix)
    print(df_total_heads_revisions.iloc[i])
    break

In [ ]:
ratios_matrices = {}

# Mapping from extracted_matrix row names to df_row names
mapping = {
    "General Heads": "proper_general_heads",
    "Entity Heads": "proper_entity_heads",
    "Relation Answer Heads": "proper_relation_answer_heads",
    "Answer Specific Heads": "proper_answer_specific_heads",
}

for transition, extracted_matrix in extracted_matrices.items():
    # Extract the source step before transition
    source_step = transition.split(" → ")[0]
    
    # Get the corresponding row from df_total_heads_revisions
    df_row = df_total_heads_revisions.loc[source_step]
    
    # Create a new matrix to store the ratios
    ratio_matrix = extracted_matrix.copy()
    
    for category in extracted_matrix.index:  # Iterate over row names
        if category in mapping:  # Ensure it's a known category
            denominator = df_row.get(mapping[category], 1)  # Use the correct total count for this head type (default to 1 to avoid division by zero)
        elif category == "Deactivated Count":
            denominator = df_row.get("total_count", 1)  # Use the overall total count for deactivated heads
        else:
            denominator = 1  # Default to 1 to avoid errors

        for col in extracted_matrix.columns:  # Iterate over columns
            if denominator != 0:
                ratio_matrix.loc[category, col] = extracted_matrix.loc[category, col] / denominator
            else:
                ratio_matrix.loc[category, col] = 0  # Set to 0 if denominator is 0

    # Store the ratio matrix
    ratios_matrices[transition] = ratio_matrix


In [ ]:
ratios_matrices

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import math

def plot_all_heatmaps(extracted_matrices):
    num_matrices = len(extracted_matrices)
    cols = 5  # Number of heatmaps per row
    rows = math.ceil(num_matrices / cols)  # Calculate required rows

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4))
    axes = axes.flatten()  # Flatten in case we don't get a perfect grid

    for i, (transition, df_matrix) in enumerate(extracted_matrices.items()):
        sns.heatmap(df_matrix, annot=True, fmt=".1f", cmap="coolwarm", linewidths=0.5, ax=axes[i])
        axes[i].set_title(f"{transition}")
        axes[i].set_xlabel("Categories")
        axes[i].set_ylabel("Head Types")

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

plot_all_heatmaps(ratios_matrices)

# Iterate over extracted matrices and plot heatmaps
# for transition, df_matrix in extracted_matrices.items():
#   plot_heatmap(df_matrix, f"Heatmap: {transition}")

In [ ]:
plot_all_heatmaps(extracted_matrices)

In [ ]:
results

In [ ]:
step5000_to_step100000_matrices

In [ ]:
ax = sns.heatmap(uniform_data, linewidth=0.5)
plt.show()
results

In [ ]:
parsed_results = parse_map_results(results)
generate_heatmap(parsed_results)

In [ ]:
results[map_type].keys()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract data for visualization
map_type = "proper_general_heads"
steps = list(results[map_type].keys())
total_heads_rev1 = [results[map_type][step]["total_heads_rev1"] for step in steps]
total_heads_rev2 = [results[map_type][step]["total_heads_rev2"] for step in steps]
deactivated_heads = [results[map_type][step]["deactivated_count"] for step in steps]
#changed_heads = [results[map_type][step]["changed"] for step in steps]
new_heads = [results[map_type][step]["new_heads"] for step in steps]
role_switch_general = [results[map_type][step]["role_switch_counts"]["general"] for step in steps]
role_switch_entity = [results[map_type][step]["role_switch_counts"]["entity"] for step in steps]
role_switch_relation_answer = [results[map_type][step]["role_switch_counts"]["relation_answer"] for step in steps]
role_switch_answer_specific = [results[map_type][step]["role_switch_counts"]["answer_specific"] for step in steps]

# Convert steps to a numeric format for plotting
step_numbers = list(range(len(steps)))

#Line plot: Total Heads
plt.figure(figsize=(12, 6))
plt.plot(step_numbers, total_heads_rev1, label="Total Heads", marker="o")
#plt.plot(step_numbers, total_heads_rev2, label="Total Heads (Rev2)", marker="o")
plt.title(f"Total Heads Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Total Heads")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

# # # Line plot: Deactivated, Changed, New Heads
# plt.figure(figsize=(12, 6))
# plt.plot(step_numbers, deactivated_heads, label="Deactivated Heads", marker="o", color="red")
# plt.plot(step_numbers, changed_heads, label="Changed Heads", marker="o", color="orange")
# plt.plot(step_numbers, new_heads, label="New Heads", marker="o", color="green")
# plt.title(f"Head Changes Over Steps ({map_type})")
# plt.xlabel("Step Transitions")
# plt.ylabel("Number of Heads")
# plt.xticks(step_numbers, steps, rotation=45, ha="right")
# plt.legend()
# plt.tight_layout()
# plt.show()

# Stacked bar chart: Role Switch Counts
width = 0.8
bar1 = np.array(role_switch_general)
bar2 = np.array(role_switch_entity)
bar3 = np.array(role_switch_relation_answer)
bar4 = np.array(role_switch_answer_specific)

plt.figure(figsize=(12, 6))
plt.bar(step_numbers, bar1, label="General Role Switches", width=width)
plt.bar(step_numbers, bar2, bottom=bar1, label="Entity Role Switches", width=width)
plt.bar(step_numbers, bar3, bottom=bar1 + bar2, label="Relation Answer Role Switches", width=width)
plt.bar(step_numbers, bar4, bottom=bar1 + bar2 + bar3, label="Answer Specific Role Switches", width=width)
plt.title(f"Role Switch Counts Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Role Switch Counts")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Extract data for visualization
map_type = "proper_entity_heads"
steps = list(results[map_type].keys())
total_heads_rev1 = [results[map_type][step]["total_heads_rev1"] for step in steps]
total_heads_rev2 = [results[map_type][step]["total_heads_rev2"] for step in steps]
deactivated_heads = [results[map_type][step]["deactivated_count"] for step in steps]
#changed_heads = [results[map_type][step]["changed"] for step in steps]
new_heads = [results[map_type][step]["new_heads"] for step in steps]
role_switch_general = [results[map_type][step]["role_switch_counts"]["general"] for step in steps]
role_switch_entity = [results[map_type][step]["role_switch_counts"]["entity"] for step in steps]
role_switch_relation_answer = [results[map_type][step]["role_switch_counts"]["relation_answer"] for step in steps]
role_switch_answer_specific = [results[map_type][step]["role_switch_counts"]["answer_specific"] for step in steps]

# Convert steps to a numeric format for plotting
step_numbers = list(range(len(steps)))

# Line plot: Total Heads
plt.figure(figsize=(12, 6))
plt.plot(step_numbers, total_heads_rev1, label="Total Heads", marker="o")
#plt.plot(step_numbers, total_heads_rev2, label="Total Heads (Rev2)", marker="o")
plt.title(f"Total Heads Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Total Heads")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

# Line plot: Deactivated, Changed, New Heads
# plt.figure(figsize=(12, 6))
# plt.plot(step_numbers, deactivated_heads, label="Deactivated Heads", marker="o", color="red")
# plt.plot(step_numbers, changed_heads, label="Changed Heads", marker="o", color="orange")
# plt.plot(step_numbers, new_heads, label="New Heads", marker="o", color="green")
# plt.title(f"Head Changes Over Steps ({map_type})")
# plt.xlabel("Step Transitions")
# plt.ylabel("Number of Heads")
# plt.xticks(step_numbers, steps, rotation=45, ha="right")
# plt.legend()
# plt.tight_layout()
# plt.show()

# Stacked bar chart: Role Switch Counts
width = 0.8
bar1 = np.array(role_switch_general)
bar2 = np.array(role_switch_entity)
bar3 = np.array(role_switch_relation_answer)
bar4 = np.array(role_switch_answer_specific)

plt.figure(figsize=(12, 6))
plt.bar(step_numbers, bar1, label="General Role Switches", width=width)
plt.bar(step_numbers, bar2, bottom=bar1, label="Entity Role Switches", width=width)
plt.bar(step_numbers, bar3, bottom=bar1 + bar2, label="Relation Answer Role Switches", width=width)
plt.bar(step_numbers, bar4, bottom=bar1 + bar2 + bar3, label="Answer Specific Role Switches", width=width)
plt.title(f"Role Switch Counts Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Role Switch Counts")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Extract data for visualization
map_type = "proper_relation_answer_heads"
steps = list(results[map_type].keys())
total_heads_rev1 = [results[map_type][step]["total_heads_rev1"] for step in steps]
total_heads_rev2 = [results[map_type][step]["total_heads_rev2"] for step in steps]
deactivated_heads = [results[map_type][step]["deactivated_count"] for step in steps]
#changed_heads = [results[map_type][step]["changed"] for step in steps]
new_heads = [results[map_type][step]["new_heads"] for step in steps]
role_switch_general = [results[map_type][step]["role_switch_counts"]["general"] for step in steps]
role_switch_entity = [results[map_type][step]["role_switch_counts"]["entity"] for step in steps]
role_switch_relation_answer = [results[map_type][step]["role_switch_counts"]["relation_answer"] for step in steps]
role_switch_answer_specific = [results[map_type][step]["role_switch_counts"]["answer_specific"] for step in steps]

# Convert steps to a numeric format for plotting
step_numbers = list(range(len(steps)))

# Line plot: Total Heads
plt.figure(figsize=(12, 6))
plt.plot(step_numbers, total_heads_rev1, label="Total Heads", marker="o")
#plt.plot(step_numbers, total_heads_rev2, label="Total Heads (Rev2)", marker="o")
plt.title(f"Total Heads Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Total Heads")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

# # Line plot: Deactivated, Changed, New Heads
# plt.figure(figsize=(12, 6))
# plt.plot(step_numbers, deactivated_heads, label="Deactivated Heads", marker="o", color="red")
# plt.plot(step_numbers, changed_heads, label="Changed Heads", marker="o", color="orange")
# plt.plot(step_numbers, new_heads, label="New Heads", marker="o", color="green")
# plt.title(f"Head Changes Over Steps ({map_type})")
# plt.xlabel("Step Transitions")
# plt.ylabel("Number of Heads")
# plt.xticks(step_numbers, steps, rotation=45, ha="right")
# plt.legend()
# plt.tight_layout()
# plt.show()

# Stacked bar chart: Role Switch Counts
width = 0.8
bar1 = np.array(role_switch_general)
bar2 = np.array(role_switch_entity)
bar3 = np.array(role_switch_relation_answer)
bar4 = np.array(role_switch_answer_specific)

plt.figure(figsize=(12, 6))
plt.bar(step_numbers, bar1, label="General Role Switches", width=width)
plt.bar(step_numbers, bar2, bottom=bar1, label="Entity Role Switches", width=width)
plt.bar(step_numbers, bar3, bottom=bar1 + bar2, label="Relation Answer Role Switches", width=width)
plt.bar(step_numbers, bar4, bottom=bar1 + bar2 + bar3, label="Answer Specific Role Switches", width=width)
plt.title(f"Role Switch Counts Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Role Switch Counts")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Extract data for visualization
map_type = "proper_answer_specific_heads"
steps = list(results[map_type].keys())
total_heads_rev1 = [results[map_type][step]["total_heads_rev1"] for step in steps]
total_heads_rev2 = [results[map_type][step]["total_heads_rev2"] for step in steps]
deactivated_heads = [results[map_type][step]["deactivated_count"] for step in steps]
#changed_heads = [results[map_type][step]["changed"] for step in steps]
new_heads = [results[map_type][step]["new_heads"] for step in steps]
role_switch_general = [results[map_type][step]["role_switch_counts"]["general"] for step in steps]
role_switch_entity = [results[map_type][step]["role_switch_counts"]["entity"] for step in steps]
role_switch_relation_answer = [results[map_type][step]["role_switch_counts"]["relation_answer"] for step in steps]
role_switch_answer_specific = [results[map_type][step]["role_switch_counts"]["answer_specific"] for step in steps]

# Convert steps to a numeric format for plotting
step_numbers = list(range(len(steps)))

# Line plot: Total Heads
plt.figure(figsize=(12, 6))
plt.plot(step_numbers, total_heads_rev1, label="Total Heads", marker="o")
#plt.plot(step_numbers, total_heads_rev2, label="Total Heads (Rev2)", marker="o")
plt.title(f"Total Heads Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Total Heads")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

# Line plot: Deactivated, Changed, New Heads
# plt.figure(figsize=(12, 6))
# plt.plot(step_numbers, deactivated_heads, label="Deactivated Heads", marker="o", color="red")
# plt.plot(step_numbers, changed_heads, label="Changed Heads", marker="o", color="orange")
# plt.plot(step_numbers, new_heads, label="New Heads", marker="o", color="green")
# plt.title(f"Head Changes Over Steps ({map_type})")
# plt.xlabel("Step Transitions")
# plt.ylabel("Number of Heads")
# plt.xticks(step_numbers, steps, rotation=45, ha="right")
# plt.legend()
# plt.tight_layout()
# plt.show()

# Stacked bar chart: Role Switch Counts
width = 0.8
bar1 = np.array(role_switch_general)
bar2 = np.array(role_switch_entity)
bar3 = np.array(role_switch_relation_answer)
bar4 = np.array(role_switch_answer_specific)

plt.figure(figsize=(12, 6))
plt.bar(step_numbers, bar1, label="General Role Switches", width=width)
plt.bar(step_numbers, bar2, bottom=bar1, label="Entity Role Switches", width=width)
plt.bar(step_numbers, bar3, bottom=bar1 + bar2, label="Relation Answer Role Switches", width=width)
plt.bar(step_numbers, bar4, bottom=bar1 + bar2 + bar3, label="Answer Specific Role Switches", width=width)
plt.title(f"Role Switch Counts Over Steps ({map_type})")
plt.xlabel("Step Transitions")
plt.ylabel("Role Switch Counts")
plt.xticks(step_numbers, steps, rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()
